# 1. Evaluación de Métodos Tabulares

En este nb evaluaremos los algoritmos de aprendizaje por refuerzo tabulares desarrollados:
- **Monte Carlo On-Policy** (Primera/Todas las visitas)
- **Monte Carlo Off-Policy** (con Importance Sampling)
- **SARSA** (TD Control On-Policy)
- **Q-Learning** (TD Control Off-Policy)

Utilizaremos el entorno `FrozenLake-v1` de Gymnasium, cuyas variables de estado y acción son discretas, ideales para alojarse en una *Q-Table*.

In [ ]:
# Instalación de dependencias (necesario en Colab)
# !pip install gymnasium pygame matplotlib seaborn tqdm

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# Importar los componentes desarrollados
from src.agents.agent import Agent
from src.policies.epsilon_greedy import EpsilonGreedyPolicy
from src.policies.random_policy import RandomPolicy
from src.learners.mc_on_policy import MCOnPolicy
from src.learners.mc_off_policy import MCOffPolicy
from src.learners.q_learning import QLearning
from src.learners.sarsa import SARSA
from src.plotting.plotting import plot_rewards, plot_episode_lengths

## Preparación del Experimento

Definimos los parámetros de entrenamiento y el entorno. FrozenLake tiene 16 estados (4x4) y 4 acciones posibles.

In [ ]:
env_name = "FrozenLake-v1"
env = gym.make(env_name, is_slippery=False) # is_slippery=False para facilitar el aprendizaje inicial

state_size = env.observation_space.n
action_size = env.action_space.n

# Parámetros generales
n_episodes = 2000
n_runs = 5 # Para sacar estadísticas más robustas
gamma = 0.99
alpha = 0.1

## Entrenamiento y Evaluación

In [ ]:
# Inicializar políticas
policy = EpsilonGreedyPolicy(epsilon_start=1.0, epsilon_min=0.01, epsilon_decay=0.995)
random_policy = RandomPolicy()

# Inicializar Learners
mc_on = MCOnPolicy(state_size, action_size, gamma)
# MC Off Policy requiere una política de comportamiento (behavior). En este caso usamos random
mc_off = MCOffPolicy(state_size, action_size, gamma, behavior_policy=random_policy)
q_learning = QLearning(state_size, action_size, alpha, gamma)
sarsa = SARSA(state_size, action_size, alpha, gamma, policy=policy)

algorithms = {
    "MC On-Policy": (mc_on, policy),
    "MC Off-Policy": (mc_off, policy), # La política target es policy, la behavior está en el learner
    "Q-Learning": (q_learning, policy),
    "SARSA": (sarsa, policy)
}

results_rewards = []
results_lengths = []
labels = []

for name, (learner, pol) in algorithms.items():
    print(f"--- Entrenando {name} ---")
    agent = Agent(env, learner, pol)
    qtable, rewards, lengths, stats = agent.train(num_episodes=n_episodes, n_runs=n_runs)
    
    results_rewards.append(rewards)
    results_lengths.append(lengths)
    labels.append(name)

## Gráficas de Resultados

In [ ]:
# Visualización usando las funciones de src/plotting
plot_rewards(results_rewards, legend_labels=labels, log_scale=False)
plot_episode_lengths(results_lengths, legend_labels=labels, log_scale=True)